# Threads: керована конкурентність

Цей ноутбук покаже, як працювати з потоками та інструментами довкола них. Розглянемо проблему гонки потоків та синхронізацій, що можуть зарадити. Подивимось на приклади з семафорами, івентами та умовами.


In [1]:
import dis
import queue
import random
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import BoundedSemaphore, Condition, Event, Semaphore, Thread

## Базовий приклад з лічильником

Починаємо з простого прикладу, де кілька потоків змінюють спільний стан.
Саме на таких прикладах легко побачити, чому синхронізація взагалі потрібна.


In [2]:
class Counter:
    def __init__(self):
        self.value = 0

    def increase(self):
        self.value += 1


def work(counter: Counter, operations_count: int):
    for _ in range(operations_count):
        counter.increase()


def run_threads(counter: Counter, count: int, operations_count: int):
    threads = []

    for _ in range(count):
        t = threading.Thread(target=work, args=(counter, operations_count))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

Це основа для кількох наступних фрагментів.
Маємо спільний `Counter`, робочу функцію `work` і запуск кількох потоків через `run_threads`.


In [3]:
threads_count = 10
operations_per_thread_count = 1_000_000

counter = Counter()
run_threads(counter, threads_count, operations_per_thread_count)

excepted_result = threads_count * operations_per_thread_count
actual_result = counter.value
print(f'Очікуваний результат: {excepted_result}, актуальний результат: {actual_result}')

Очікуваний результат: 10000000, актуальний результат: 10000000


Тут ідея проста: очікуємо, що значення дорівнюватиме кількості всіх інкрементів.
На що звернути увагу: в темі потоків навіть такий простий код не завжди поводиться так, як очікуємо.


## Bash-скрипт із лекції

Цей скрипт допомагає перевірити той самий приклад на різних версіях Python.
Це корисно, коли хочемо порівняти поведінку потоків у різних інтерпретаторах.

```bash
#!/bin/bash

PYTHON_VERSIONS=(3.8 3.9 3.10 3.11 3.12)
VENV_DIR="./venvs"

mkdir -p "$VENV_DIR"

for VERSION in "${PYTHON_VERSIONS[@]}"; do
    PYTHON_BIN="python$VERSION"
    VENV_PATH="$VENV_DIR/$VERSION"

    if ! command -v "$PYTHON_BIN" >/dev/null 2>&1; then
        echo "Skip Python $VERSION: $PYTHON_BIN not found"
        continue
    fi

    echo "Run with Python $VERSION"

    "$PYTHON_BIN" -m venv "$VENV_PATH"

    "$VENV_PATH/bin/python" --version
    "$VENV_PATH/bin/python" test.py
done
```

Його варто запускати з термінала: скрипт створює окремі середовища, показує версію Python і запускає тестовий файл.


## `dis`: дивимось, що реально виконує Python

Переходимо до дослідження внутрішньої механіки.
`dis` допомагає побачити, що навіть короткий вираз складається з кількох кроків.


In [4]:
dis.dis(Counter().increase)
dis.dis(work)

  5           0 RESUME                   0

  6           2 LOAD_FAST                0 (self)
              4 COPY                     1
              6 LOAD_ATTR                0 (value)
             26 LOAD_CONST               1 (1)
             28 BINARY_OP               13 (+=)
             32 SWAP                     2
             34 STORE_ATTR               0 (value)
             44 RETURN_CONST             0 (None)
  9           0 RESUME                   0

 10           2 LOAD_GLOBAL              1 (NULL + range)
             12 LOAD_FAST                1 (operations_count)
             14 CALL                     1
             22 GET_ITER
        >>   24 FOR_ITER                18 (to 64)
             28 STORE_FAST               2 (_)

 11          30 LOAD_FAST                0 (counter)
             32 LOAD_ATTR                3 (NULL|self + increase)
             52 CALL                     0
             60 POP_TOP
             62 JUMP_BACKWARD           20 (to 24)

 10 

Цей приклад показує, що `self.value += 1` не є однією неподільною дією. Але, починаючи з версії python3.10 інкремент було зроблено атомарною операцією в сенсі відсутності байткод-інструкцій, що можуть (але не обов’язково) викликати звільнення GIL. Це JUMP_BACKWARD (або JUMP_ABSOLUTE, в залежності від версій Python), яка виконує перехід на початок циклу і може перевіряти GIL у довгих циклах, або CALL (у версіях до Python 3.11 - CALL_FUNCTION), що може звільнити GIL у випадках, коли викликається функція C, I/O або складні обчислення.

Чому це важливо: між цими кроками планувальник може переключити виконання на інший потік у випадку, якщо під час дизасемблювання ви бачите байткод-інструкції переривання.


## Варіанти зміни `increase` для профілювання через `dis`

Нижче збережено фрагменти з інкрементами, де навіть після python3.10 побачимо гонку потоків через присутні байткод-інструкціх переривання. Можна замінити звичайний інкремент у прикладі вище цими варіантами та попрофілювати на різних версіях Python.


In [5]:
def increase_abs(self):
    self.value += abs(1)


def increase_int(self):
    self.value += int(1)


def increase_round(self):
    self.value += round(1)


def increase_lambda(self):
    self.value += (lambda x: x + 1)(0)


dis.dis(increase_abs)
dis.dis(increase_int)
dis.dis(increase_round)
dis.dis(increase_lambda)

  1           0 RESUME                   0

  2           2 LOAD_FAST                0 (self)
              4 COPY                     1
              6 LOAD_ATTR                0 (value)
             26 LOAD_GLOBAL              3 (NULL + abs)
             36 LOAD_CONST               1 (1)
             38 CALL                     1
             46 BINARY_OP               13 (+=)
             50 SWAP                     2
             52 STORE_ATTR               0 (value)
             62 RETURN_CONST             0 (None)
  5           0 RESUME                   0

  6           2 LOAD_FAST                0 (self)
              4 COPY                     1
              6 LOAD_ATTR                0 (value)
             26 LOAD_GLOBAL              3 (NULL + int)
             36 LOAD_CONST               1 (1)
             38 CALL                     1
             46 BINARY_OP               13 (+=)
             50 SWAP                     2
             52 STORE_ATTR               0 (value

## Той самий приклад, але з `threading.Lock()`

Спробуймо виправити ситуацію за допомогою `threading.Lock()`

In [6]:
class Counter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()

    def increase(self):
        with self.lock:
            self.value += int(1)


def work(counter: Counter, operations_count: int):
    for _ in range(operations_count):
        counter.increase()


def run_threads(counter: Counter, count: int, operations_count: int):
    threads = []

    for _ in range(count):
        t = threading.Thread(target=work, args=(counter, operations_count))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

Перевіривши запуск на різних версіях Python знову, побачимо, що синхронізація за допомогою `lock` прибрала проблему гонки потоків, і це вже правильніша модель роботи зі спільними даними.


## Погане використання `Lock`

Тепер дослідимо не лише наявність блокування, а й те, як довго його тримати.
Це дуже практичний момент, бо правильний код може бути або швидшим, або повільнішим залежно від межі критичної секції.


In [7]:
shared_data: list[str] = []
lock = threading.Lock()


def save_bad(name: str) -> None:
    with lock:
        time.sleep(0.5)  # імітація повільного I/O
        shared_data.append(name)


def save_good(name: str) -> None:
    time.sleep(0.5)  # I/O поза lock
    with lock:
        shared_data.append(name)


def run(func, label: str) -> None:
    shared_data.clear()
    threads = [threading.Thread(target=func, args=(f'T{i}',)) for i in range(4)]

    start = time.perf_counter()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    elapsed = time.perf_counter() - start

    print(f'{label}: {elapsed:.2f}s -> {shared_data}')


run(save_bad, 'Bad locking')
run(save_good, 'Good locking')

Bad locking: 2.02s -> ['T0', 'T1', 'T2', 'T3']
Good locking: 0.51s -> ['T0', 'T1', 'T2', 'T3']


Що тут відбувається: у першому варіанті повільна операція лежить всередині `lock`, а в другому поза ним.
Чому це важливо: блокування треба тримати тільки стільки, скільки потрібно для захисту спільних даних.


## Deadlock - проблема взаємного блокування

Цей приклад показує deadlock: `outer_method()` бере `Lock`, а `inner_method()` намагається взяти той самий `Lock` ще раз.
У результаті потік застрягає в очікуванні самого себе.

```python
import threading

class DeadLockExample:
    def __init__(self):
        self.lock = threading.Lock()

    def outer_method(self):
        with self.lock:
            print('Зовнішній метод залоковано')
            self.inner_method()

    def inner_method(self):
        with self.lock:
            print('Внутрішній метод залоковано')

example = DeadLockExample()
thread = threading.Thread(target=example.outer_method)
thread.start()
thread.join()
```


## `RLock` - "розумний" lock, або як вирішити проблему Deadlock

Подивимось різницю між звичайним `Lock` і повторно-вхідним `RLock`.
Часто зручніше та краще писати `with lock:`, але важливо розуміти і ручні `acquire()` / `release()`.
Це та сама базова механіка, тільки без контекстного менеджера.


In [8]:
lock = threading.Lock()

lock.acquire()  # Захоплення блокування
print('Lock acquired')
lock.release()  # Звільнення блокування
print('Lock released')


rlock = threading.RLock()

rlock.acquire()  # Захоплюємо 1-й раз
print('RLock acquired first time')
rlock.acquire()  # Захоплюємо 2-й раз
print('RLock acquired second time')
rlock.release()  # Перший `release`
print('RLock released once')
rlock.release()  # Другий `release`, тепер повністю розблоковано
print('RLock fully released')

Lock acquired
Lock released
RLock acquired first time
RLock acquired second time
RLock released once
RLock fully released


`RLock` потрібен тоді, коли один і той самий потік може зайти в захищений код повторно.
Без цього деякі вкладені виклики легко приводять до deadlock.


Можемо виправити минулий приклад, замінивши `Lock` на `RLock`.

In [9]:
import threading


class DeadLockExample:
    def __init__(self):
        self.lock = threading.RLock()

    def outer_method(self):
        with self.lock:
            print('Зовнішній метод залоковано')
            self.inner_method()

    def inner_method(self):
        with self.lock:
            print('Внутрішній метод залоковано')


example = DeadLockExample()
thread = threading.Thread(target=example.outer_method)
thread.start()
thread.join()

Зовнішній метод залоковано
Внутрішній метод залоковано


## `ProducerConsumer` через `queue.Queue`

Погляньмо на приклад координації між потоками за допомогою черги.
Тут один потік додає елементи в чергу, а інші потоки їх обробляють.


In [10]:
class ProducerConsumer:
    def __init__(self, num_consumers: int = 5, num_elements: int = 10):
        self.q = queue.Queue()
        self.producer_thread = threading.Thread(target=self._producer)
        self.consumer_threads = [threading.Thread(target=self._consumer) for _ in range(num_consumers)]
        self.num_elements = num_elements

    def _producer(self):
        for i in range(self.num_elements):
            self.q.put(i)  # Додає елемент у чергу
            print(f'Додано {i}')
            time.sleep(0.1)  # Імітація роботи

    def _consumer(self):
        while not self.q.empty():
            item = self.q.get()  # Отримує елемент
            print(f'Отримано {item}')
            self.q.task_done()  # Позначає елемент як оброблений
            time.sleep(0.2)  # Імітація обробки

    def run(self):
        self.producer_thread.start()
        time.sleep(0.5)  # Даємо продюсеру час додати дані

        for t in self.consumer_threads:
            t.start()

        self.producer_thread.join()  # Чекаємо завершення продюсера

        self.q.join()  # Чекаємо, поки всі елементи будуть оброблені
        for t in self.consumer_threads:
            t.join()


pc = ProducerConsumer()
pc.run()

Додано 0
Додано 1
Додано 2
Додано 3
Додано 4
Отримано 0
Отримано 1
Отримано 2
Отримано 3
Отримано 4
Додано 5
Додано 6
Отримано 5
Отримано 6
Додано 7
Додано 8
Отримано 7
Отримано 8
Додано 9
Отримано 9


`Queue` допомагає безпечно передавати роботу між потоками.
Тут один потік додає елементи, а кілька інших по черзі їх забирають і обробляють.


## Ще один варіант producer-consumer з `Queue`

Це компактніший приклад із sentinel-значенням `None`, яке каже споживачу зупинитися.


In [11]:
task_queue: queue.Queue[int | None] = queue.Queue(maxsize=2)


def producer() -> None:
    for item in range(5):
        print(f'Producing {item}')
        task_queue.put(item)
        print(f'Queued {item}')
    task_queue.put(None)


def consumer() -> None:
    while True:
        item = task_queue.get()
        try:
            if item is None:
                print('Consumer stops')
                break

            print(f'Consuming {item}')
            time.sleep(0.4)
        finally:
            task_queue.task_done()


producer_thread = threading.Thread(target=producer)
consumer_thread = threading.Thread(target=consumer)

consumer_thread.start()
producer_thread.start()

producer_thread.join()
task_queue.join()
consumer_thread.join()

Producing 0
Queued 0
Producing 1
Queued 1
Producing 2
Consuming 0
Queued 2
Producing 3
Consuming 1Queued 3
Producing 4

Consuming 2Queued 4

Consuming 3
Consuming 4
Consumer stops


Тут добре видно, як `Queue(maxsize=2)` ще й обмежує буфер між потоками.
Це корисно, коли не хочемо, щоб producer занадто швидко накопичував задачі.


## `Semaphore` - балансувальник навантаження

Семафор потрібен не для взаємовиключення одного потоку (тобто це не мʼютекс!), а для обмеження кількості одночасних доступів.


In [23]:
def worker(name: str):
    with sem:
        print(f'{name} отримав доступ')
        time.sleep(1)
        print(f'{name} звільнив доступ')


sem = threading.Semaphore(2)  # Дозволяє доступ лише 2 потокам одночасно
threads = [threading.Thread(target=worker, args=(f'Потік-{i}',)) for i in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

Потік-0 отримав доступ
Потік-1 отримав доступ
Потік-0 звільнив доступ
Потік-2 отримав доступ
Потік-1 звільнив доступ
Потік-3 отримав доступ
Потік-2 звільнив доступПотік-3 звільнив доступ
Потік-4 отримав доступ

Потік-4 звільнив доступ


На що звернути увагу: потоків п’ять, але одночасно всередині секції працюють лише два.
Це типова модель для обмежених ресурсів.


## `Semaphore` + `ThreadPoolExecutor`: обмеження паралельних завантажень

Тут уже приклад ближчий до реального застосування.
Є пул потоків, але паралельність додатково обмежується семафором.


In [24]:
random.seed(42)


def fake_download(url: str) -> str:
    print(f'Start downloading: {url}')

    time.sleep(random.uniform(0.3, 1.5))

    if 'bad' in url:
        raise RuntimeError('Download failed')

    print(f'Finish downloading: {url}')
    return f'content from {url}'


def download_worker(
    index: int,
    url: str,
    semaphore: Semaphore,
) -> tuple[int, str]:
    try:
        with semaphore:
            result = fake_download(url)
            return index, result
    except RuntimeError:
        print(f'Failed downloading: {url}')
        return index, 'ERROR'


def download_all(
    urls: list[str],
    max_concurrent_requests: int,
) -> list[str]:
    semaphore = Semaphore(max_concurrent_requests)
    results = [''] * len(urls)

    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = [executor.submit(download_worker, index, url, semaphore) for index, url in enumerate(urls)]

        for future in futures:
            index, result = future.result()
            results[index] = result

    return results


urls = [
    'https://api.example.com/users',
    'https://api.example.com/orders',
    'https://api.example.com/products',
    'https://api.example.com/bad-endpoint',
    'https://api.example.com/reviews',
    'https://api.example.com/comments',
    'https://api.example.com/profile',
]

result = download_all(urls, max_concurrent_requests=3)

print()
print('Result:')
for index, value in enumerate(result):
    print(index, value)

Start downloading: https://api.example.com/users
Start downloading: https://api.example.com/orders
Start downloading: https://api.example.com/products
Finish downloading: https://api.example.com/orders
Start downloading: https://api.example.com/bad-endpoint
Finish downloading: https://api.example.com/products
Start downloading: https://api.example.com/reviews
Failed downloading: https://api.example.com/bad-endpointStart downloading: https://api.example.com/comments

Finish downloading: https://api.example.com/users
Start downloading: https://api.example.com/profile
Finish downloading: https://api.example.com/reviews
Finish downloading: https://api.example.com/comments
Finish downloading: https://api.example.com/profile

Result:
0 content from https://api.example.com/users
1 content from https://api.example.com/orders
2 content from https://api.example.com/products
3 ERROR
4 content from https://api.example.com/reviews
5 content from https://api.example.com/comments
6 content from https

Чому це важливо: навіть якщо потоків багато, зовнішній ресурс може не витримати занадто багато одночасних звернень.
Семафор тут контролює саме навантаження на ресурс.


## `BoundedSemaphore`: модель пулу з’єднань до бази

Ще один практичний приклад, де треба обмежити кількість одночасних операцій.
Тут роль ресурсу виконує пул доступних з’єднань.


In [25]:
random.seed(42)


class DatabaseConnectionPool:
    def __init__(self, size: int) -> None:
        self._available_connections = BoundedSemaphore(value=size)

    def execute_query(self, query: str) -> str:
        with self._available_connections:
            print(f'Start query: {query}')

            time.sleep(random.uniform(0.5, 1.5))

            print(f'Finish query: {query}')
            return f'result for {query}'


def run_query(
    pool: DatabaseConnectionPool,
    query: str,
) -> str:
    return pool.execute_query(query)


def run_queries(
    queries: list[str],
    pool_size: int,
    max_workers: int,
) -> list[str]:
    pool = DatabaseConnectionPool(size=pool_size)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(run_query, pool, query) for query in queries]

        return [future.result() for future in futures]


queries = [
    'SELECT * FROM users',
    'SELECT * FROM orders',
    'SELECT * FROM products',
    'SELECT * FROM payments',
    'SELECT * FROM reviews',
    'SELECT * FROM sessions',
    'SELECT * FROM invoices',
]

results = run_queries(
    queries=queries,
    pool_size=3,
    max_workers=10,
)

print()
print('Results:')
for result in results:
    print(result)

Start query: SELECT * FROM users
Start query: SELECT * FROM orders
Start query: SELECT * FROM products
Finish query: SELECT * FROM orders
Start query: SELECT * FROM payments
Finish query: SELECT * FROM products
Start query: SELECT * FROM reviews
Finish query: SELECT * FROM users
Start query: SELECT * FROM sessions
Finish query: SELECT * FROM payments
Start query: SELECT * FROM invoices
Finish query: SELECT * FROM reviews
Finish query: SELECT * FROM sessions
Finish query: SELECT * FROM invoices

Results:
result for SELECT * FROM users
result for SELECT * FROM orders
result for SELECT * FROM products
result for SELECT * FROM payments
result for SELECT * FROM reviews
result for SELECT * FROM sessions
result for SELECT * FROM invoices


Це вже дуже схоже на реальну інженерну задачу.
`BoundedSemaphore` тут допомагає не “створити зайві з’єднання”, а працювати в межах доступного пулу.


## `Condition`: очікування готовності даних

Тут один потік чекає, поки інший підготує дані і подасть сигнал.
`Condition` корисний тоді, коли важливий не просто доступ до спільного об’єкта, а певна подія або стан.


In [15]:
class DataPipeline:
    def __init__(self):
        self.data_ready = False
        self.condition = threading.Condition()

    def produce(self):
        with self.condition:
            print('Producer генерує дані')
            time.sleep(1)
            self.data_ready = True
            self.condition.notify_all()  # Сповіщає всі потоки, що дані готові

    def consume(self):
        with self.condition:
            while not self.data_ready:
                self.condition.wait()  # Чекає на `notify_all()`
            print('Consumer отримав дані')


pipeline = DataPipeline()

consumer_thread = threading.Thread(target=pipeline.consume)
producer_thread = threading.Thread(target=pipeline.produce)
consumer_thread.start()
producer_thread.start()
consumer_thread.join()
producer_thread.join()

Producer генерує дані
Consumer отримав дані


Тут споживач не робить зайвих перевірок у циклі, а спокійно чекає на сигнал.
Це зручний спосіб координувати залежні кроки між потоками.


## `Condition`: можливості timeout через `wait_for`

Цей варіант показує, як чекати на подію не нескінченно, а з обмеженням у часі.
Якщо дані не з’являться вчасно, потік зможе коректно завершити очікування.


In [16]:
class DataPipelineWithTimeout:
    def __init__(self):
        self.data_ready = False
        self.condition = threading.Condition()

    def produce(self):
        print('Producer генерує дані повільніше')
        time.sleep(3)
        with self.condition:
            self.data_ready = True
            self.condition.notify_all()

    def consume(self):
        with self.condition:
            ready = self.condition.wait_for(lambda: self.data_ready, timeout=2.0)
            if not ready:
                print('Consumer: timeout, дані не з’явилися вчасно')
                return
            print('Consumer отримав дані')


pipeline = DataPipelineWithTimeout()

consumer_thread = threading.Thread(target=pipeline.consume)
producer_thread = threading.Thread(target=pipeline.produce)
consumer_thread.start()
time.sleep(0.1)
producer_thread.start()
consumer_thread.join()
producer_thread.join()

Producer генерує дані повільніше
Consumer: timeout, дані не з’явилися вчасно


`wait_for(..., timeout=2.0)` дає змогу чекати не просто на сигнал, а на конкретний стан і лише певний час.
Це зручно, коли система має вміти вчасно зреагувати на затримку, а не зависати в очікуванні.


## `Condition` у прикладі з конфігурацією

Ще один приклад на основі `Condition` показує стартову синхронізацію між потоками.
Один потік завантажує конфігурацію, а інший чекає, поки ці дані стануть доступними.


In [17]:
class ConfigStore:
    def __init__(self) -> None:
        self._condition = Condition()
        self._config: dict[str, str] | None = None

    def load_config(self) -> None:
        print('Loading config...')
        time.sleep(2)

        config = {
            'database_url': 'postgresql://localhost:5432/app',
            'log_level': 'INFO',
        }

        with self._condition:
            self._config = config
            print('Config loaded')
            self._condition.notify_all()

    def get_config(self) -> dict[str, str]:
        with self._condition:
            while self._config is None:
                print('Waiting for config...')
                self._condition.wait()

            return self._config


def worker(store: ConfigStore) -> None:
    config = store.get_config()
    print(f'Worker started with config: {config}')


store = ConfigStore()

loader_thread = Thread(target=store.load_config)
worker_thread = Thread(target=worker, args=(store,))

worker_thread.start()
loader_thread.start()

worker_thread.join()
loader_thread.join()

Waiting for config...
Loading config...
Config loaded
Worker started with config: {'database_url': 'postgresql://localhost:5432/app', 'log_level': 'INFO'}


Тут уже легко уявити реальну систему: спочатку приходять налаштування, а вже потім починається робота.
`Condition` добре підходить для таких сценаріїв стартової синхронізації.


## `Event`: простий сигнал між потоками

`Event` зручний тоді, коли один потік має просто дочекатися дозволу або сигналу від іншого.
Це одна з найпростіших форм координації між потоками.


In [18]:
event = Event()


def worker():
    print('Очікую на сигнал')
    event.wait()  # Чекає на set()
    print('Потік запущено!')


thread = threading.Thread(target=worker)
thread.start()

time.sleep(2)
event.set()  # Дає дозвіл потоку працювати
thread.join()

Очікую на сигнал
Потік запущено!


Тут потік нічого не робить, поки не отримає сигнал через `set()`.
На що звернути увагу: `Event` не передає дані, а лише повідомляє, що певна подія вже сталася.


## `Event` у сценарії старту застосунку

Цей приклад показує, як кілька воркерів можуть чекати, поки система завершить початкову підготовку.
Після сигналу всі потоки переходять до обробки задач.


In [19]:
def startup_app(app_ready: Event) -> None:
    print('Loading config...')
    time.sleep(1)

    print('Connecting to database...')
    time.sleep(1)

    print('Warming up cache...')
    time.sleep(1)

    print('Application is ready')
    app_ready.set()


def event_worker(
    worker_name: str,
    app_ready: Event,
    tasks: queue.Queue[str],
) -> None:
    print(f'{worker_name}: waiting for application startup')

    app_ready.wait()

    print(f'{worker_name}: started processing tasks')

    while True:
        try:
            task = tasks.get(timeout=1)
        except queue.Empty:
            print(f'{worker_name}: no more tasks, stopping')
            return

        print(f'{worker_name}: processing {task}')
        time.sleep(random.uniform(0.5, 1.5))
        tasks.task_done()


app_ready = Event()
tasks: queue.Queue[str] = queue.Queue()

for task in [
    'send welcome email',
    'generate invoice',
    'sync customer profile',
    'update search index',
    'publish analytics event',
]:
    tasks.put(task)

workers = [
    Thread(
        target=event_worker,
        args=(f'worker-{index}', app_ready, tasks),
    )
    for index in range(3)
]

startup_thread = Thread(target=startup_app, args=(app_ready,))

for thread in workers:
    thread.start()

startup_thread.start()

startup_thread.join()
tasks.join()

for thread in workers:
    thread.join()

print('All tasks processed')

worker-0: waiting for application startup
worker-1: waiting for application startup
worker-2: waiting for application startup
Loading config...
Connecting to database...
Warming up cache...
Application is ready
worker-0: started processing tasks
worker-0: processing send welcome email
worker-2: started processing tasks
worker-2: processing generate invoice
worker-1: started processing tasks
worker-1: processing sync customer profile
worker-1: processing update search index
worker-0: processing publish analytics event
worker-2: no more tasks, stopping
worker-1: no more tasks, stopping
worker-0: no more tasks, stopping
All tasks processed


Тут `Event` допомагає відокремити етап ініціалізації від етапу обробки задач.
Це корисно, коли воркери не повинні стартувати раніше, ніж застосунок стане готовим до роботи.


## Від `map` до `ThreadPoolExecutor.map`

Спочатку корисно згадати звичайний `map`, а потім перейти до його паралельного варіанту в пулі потоків.
Ідея в обох випадках однакова: застосувати одну функцію до набору значень.


In [20]:
def double(x):
    return x * 2


numbers = [1, 2, 3, 4]
result = map(double, numbers)
print(list(result))  # [2, 4, 6, 8]

[2, 4, 6, 8]


Це звичайний послідовний `map`.
Він допомагає побачити знайому модель перед переходом до пулу потоків.


In [21]:
def task(n: int) -> int:
    return n * 2


with ThreadPoolExecutor(max_workers=3) as executor:
    results = executor.map(task, range(5))

print(list(results))  # [0, 2, 4, 6, 8]

[0, 2, 4, 6, 8]


`ThreadPoolExecutor.map` зручний тоді, коли треба паралельно прогнати багато однотипних задач.
Синтаксис схожий на звичайний `map`, але виконання вже відбувається через пул потоків.


## `ThreadPoolExecutor.submit` і `as_completed`

Якщо потрібно більше контролю над окремими задачами, зручно працювати через `submit()`.
А `as_completed()` дозволяє обробляти результати в тому порядку, у якому задачі реально завершуються.


In [22]:
def fetch(url_id: int) -> str:
    delay = random.uniform(0.2, 1.0)
    time.sleep(delay)

    if url_id == 3:
        raise RuntimeError('Request failed')

    return f'response-{url_id} in {delay:.2f}s'


with ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(fetch, i) for i in range(5)]

    for future in as_completed(futures):
        try:
            print('Result:', future.result())
        except Exception as exc:
            print('Error :', exc)

Result: response-0 in 0.22s
Result: response-1 in 0.36s
Result: response-2 in 0.72s
Result: response-4 in 0.38s
Error : Request failed


Тут уже видно різницю: результати приходять не за порядком `0, 1, 2...`, а за фактичним завершенням задач.
Це особливо корисно, коли окремі операції можуть падати або тривати різний час.


## Підсумок

Потоки корисні там, де важливо виконувати кілька дій паралельно, але разом з цим з’являється потреба в координації.
Саме тому в практиці так часто використовують `Lock`, `RLock`, `Queue`, `Semaphore`, `Condition`, `Event` і `ThreadPoolExecutor`.
